In [1]:
import os
import sys
import asyncio
from pathlib import Path
from dotenv import load_dotenv

# Xác định thư mục gốc và thêm vào sys.path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Load .env
load_dotenv()

# ==========================================
# KHỞI TẠO SETTINGS & HYBRID RETRIEVER
# ==========================================
from ai_agent.infrastructure.config import Settings
from ai_agent.rag.hybrid_retrieval import HybridRetriever

settings = Settings()
retriever = HybridRetriever(settings=settings)
print("✅ HybridRetriever initialized (async-ready)")

# ==========================================
# 1. KHỞI TẠO LLM (DeepSeek v4 Pro)
# ==========================================
from ai_agent.infrastructure.llm_router import get_llm
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def init_llm():
    """Khởi tạo DeepSeek v4 Pro LLM"""
    print("🤖 Mode: DeepSeek v4")
    llm = get_llm(settings.deepseek_model_default)
    return llm

llm = init_llm()

# ==========================================
# 2. ĐỊNH NGHĨA PROMPT TEMPLATE
# ==========================================
prompt_template = """
Bạn là trợ lý pháp lý chuyên về luật giao thông Việt Nam. 
Dựa vào các ngữ cảnh (context) dưới đây được trích xuất từ các văn bản luật, hãy trả lời câu hỏi của người dùng.

Yêu cầu:
- Trả lời chính xác, rõ ràng, dựa hoàn toàn vào ngữ cảnh.
- Nếu ngữ cảnh không đủ thông tin, hãy nói "Theo tài liệu hiện có, không đủ thông tin để trả lời".
- Trích dẫn nguồn (điều, khoản, tên file) sau mỗi ý quan trọng.

Ngữ cảnh:
{context}

Câu hỏi: {question}

Trả lời:
"""

prompt = ChatPromptTemplate.from_template(prompt_template)

# ==========================================
# 3. HÀM ĐỊNH DẠNG NGỮ CẢNH TỪ KẾT QUẢ RETRIEVAL
# ==========================================
def format_docs_as_context(documents):
    """Chuyển danh sách Document LangChain thành chuỗi ngữ cảnh với trích dẫn"""
    if not documents:
        return "Không có tài liệu liên quan."
    
    contexts = []
    for i, doc in enumerate(documents, 1):
        article = doc.metadata.get("article", "Không rõ điều")
        file_name = doc.metadata.get("file_name", "Không rõ nguồn")
        text = doc.page_content.strip()
        contexts.append(f"[{i}] (Điều {article}, {file_name})\n{text}\n")
    return "\n".join(contexts)

# ==========================================
# 4. REWRITE QUERY
# ==========================================
rewrite_prompt_template = """Bạn là chuyên gia về luật giao thông Việt Nam. Hãy viết lại câu hỏi của người dùng thành một câu hỏi tìm kiếm sử dụng đúng thuật ngữ pháp lý và cấu trúc phù hợp để tra cứu trong văn bản luật.

Ví dụ: "xe máy vượt đèn đỏ bị phạt bao nhiêu?" -> "Mức phạt đối với hành vi không chấp hành tín hiệu giao thông của xe máy theo luật giao thông Việt Nam là gì?"

Câu hỏi gốc: {original_query}

Câu hỏi đã viết lại (chỉ trả lời câu hỏi đã rewrite, không thêm lời dẫn):
"""

rewrite_prompt = ChatPromptTemplate.from_template(rewrite_prompt_template)

def rewrite_query(original_query: str, llm) -> str:
    """Dùng LLM để rewrite user query thành dạng chuẩn pháp lý"""
    chain = rewrite_prompt | llm | StrOutputParser()
    rewritten = chain.invoke({"original_query": original_query})
    return rewritten.strip()

# ==========================================
# 5. HÀM RAG CHÍNH (ASYNC)
# ==========================================
async def rag_answer(query: str, top_k: int = 5, do_rewrite: bool = True) -> dict:
    """
    RAG Pipeline: Rewrite → Retrieve → Generate
    """
    print(f"\n🔍 Original query: {query}")
    print("-" * 50)
    
    # 0. Rewrite query
    if do_rewrite:
        rewritten_q = await asyncio.to_thread(rewrite_query, query, llm)
        print(f"✍️ Rewritten query: {rewritten_q}")
        search_query = rewritten_q
    else:
        search_query = query
    
    # 1. Retrieve (async)
    retrieved_docs = await retriever.search(search_query, top_k=top_k)
    if not retrieved_docs:
        return {
            "query": query,
            "retrieved_docs": [],
            "context": "",
            "answer": "Không tìm thấy tài liệu liên quan trong cơ sở dữ liệu."
        }
    
    # 2. Format context
    context = format_docs_as_context(retrieved_docs)
    print(f"📄 Retrieved {len(retrieved_docs)} documents, context length: {len(context)} chars")
    
    # 3. Generate answer
    chain = prompt | llm | StrOutputParser()
    answer = await asyncio.to_thread(chain.invoke, {"context": context, "question": query})
    
    return {
        "query": query,
        "rewritten_query": search_query if do_rewrite else None,
        "retrieved_docs": retrieved_docs,
        "context": context,
        "answer": answer
    }

# ==========================================
# 6. HIỂN THỊ KẾT QUẢ ĐẸP
# ==========================================
def print_rag_result(result: dict):
    print("\n" + "="*70)
    print(f"💬 Câu hỏi: {result['query']}")
    if result.get("rewritten_query"):
        print(f"✍️  (Rewrite: {result['rewritten_query']})")
    print("="*70)
    print("\n📖 Câu trả lời:")
    print(result["answer"])
    
    print("\n📚 Nguồn tham khảo:")
    for i, doc in enumerate(result["retrieved_docs"], 1):
        meta = doc.metadata
        article = meta.get("article", "Không rõ điều")
        file_name = meta.get("file_name", "Không rõ nguồn")
        chunk_id = meta.get("chunk_id", "")
        print(f"  [{i}] {article}\n      📄 {file_name} (chunk: {chunk_id})")
    
    print("\n" + "="*70)

print("\n✅ RAG Pipeline setup complete!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ HybridRetriever initialized (async-ready)
🤖 Mode: DeepSeek v4

✅ RAG Pipeline setup complete!


In [2]:
# ==========================================
# 7. VÍ DỤ SỬ DỤNG (INTERACTIVE - ASYNC)
# ==========================================

async def interactive_rag_pipeline():
    """Interactive RAG pipeline"""
    print("\n" + "="*70)
    print("🔍 RAG PIPELINE - INTERACTIVE MODE")
    print("   Type 'exit' or 'quit' to quit")
    print("="*70)
    
    while True:
        query = input("\n🔎 Nhập câu hỏi (hoặc 'exit'): ").strip()
        if query.lower() in ('exit', 'quit', 'thoát'):
            print("👋 Goodbye!")
            break
        if not query:
            continue
        
        try:
            result = await rag_answer(query, top_k=5, do_rewrite=True)
            print_rag_result(result)
        except Exception as e:
            print(f"❌ Error: {e}")
            import traceback
            traceback.print_exc()

# Run the pipeline
if __name__ == "__main__":
    await interactive_rag_pipeline()
else:
    print("\n✅ Use: await interactive_rag_pipeline() to start interactive mode")
    print("   Or: result = await rag_answer('your question') for programmatic use")


🔍 RAG PIPELINE - INTERACTIVE MODE
   Type 'exit' or 'quit' to quit

🔍 Original query: Không chấp hành đèn tín hiệu giao thông phạt bao nhiêu
--------------------------------------------------
✍️ Rewritten query: Mức phạt đối với hành vi không chấp hành tín hiệu đèn giao thông theo quy định của pháp luật giao thông đường bộ Việt Nam là bao nhiêu?
📄 Retrieved 5 documents, context length: 5499 chars

💬 Câu hỏi: Không chấp hành đèn tín hiệu giao thông phạt bao nhiêu
✍️  (Rewrite: Mức phạt đối với hành vi không chấp hành tín hiệu đèn giao thông theo quy định của pháp luật giao thông đường bộ Việt Nam là bao nhiêu?)

📖 Câu trả lời:
Căn cứ vào Nghị định số 168/2024/NĐ-CP, mức phạt đối với hành vi không chấp hành hiệu lệnh của đèn tín hiệu giao thông phụ thuộc vào đối tượng vi phạm cụ thể như sau:

- **Người đi bộ:** Phạt tiền từ 150.000 đồng đến 250.000 đồng (điểm b khoản 1 Điều 10 - Xử phạt người đi bộ vi phạm quy tắc giao thông đường bộ).

- **Người điều khiển, dẫn dắt vật nuôi, điều khiển

In [ ]:
# ==========================================
# 7. VÍ DỤ SỬ DỤNG (INTERACTIVE - ASYNC)
# ==========================================

async def interactive_rag_pipeline():
    """Interactive RAG pipeline"""
    print("\n" + "="*70)
    print("🔍 RAG PIPELINE - INTERACTIVE MODE")
    print("   Type 'exit' or 'quit' to quit")
    print("="*70)
    
    while True:
        query = input("\n🔎 Nhập câu hỏi (hoặc 'exit'): ").strip()
        if query.lower() in ('exit', 'quit', 'thoát'):
            print("👋 Goodbye!")
            break
        if not query:
            continue
        
        try:
            result = await rag_answer(query, top_k=5, do_rewrite=True)
            print_rag_result(result)
        except Exception as e:
            print(f"❌ Error: {e}")
            import traceback
            traceback.print_exc()

# Run the pipeline
if __name__ == "__main__":
    asyncio.run(interactive_rag_pipeline())
else:
    print("\n✅ Use: asyncio.run(interactive_rag_pipeline()) to start interactive mode")
    print("   Or: result = await rag_answer('your question') for programmatic use")


🔍 Original query: Hành vi điều khiển xe máy chở hai người có được phép theo quy định của pháp luật giao thông đường bộ Việt Nam không?
--------------------------------------------------
✍️ Rewritten query: "Hành vi điều khiển xe mô tô hai bánh chở theo hai người có được phép theo quy định của Luật Giao thông đường bộ Việt Nam không?"

🔍 Searching: '"Hành vi điều khiển xe mô tô hai bánh chở theo hai người có được phép theo quy định của Luật Giao thông đường bộ Việt Nam không?"'
----------------------------------------------------------------------

🔧 Building Hybrid Ensemble Retriever

[1] Creating BM25Retriever from CHILD chunks...
✅ Loaded 3112 CHILD chunks for BM25 indexing
   ✅ BM25Retriever ready (3112 child docs)

[2] Creating VectorStoreRetriever (Qdrant)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

   ✅ VectorStoreRetriever ready

[3] Creating EnsembleRetriever (BM25 + Vector)...
   ✅ EnsembleRetriever ready (RRF fusion)

[4] Setting up CohereRerank...

[5] Creating final ContextualCompressionRetriever...
   ✅ Final retriever ready (top_k=5)

✅ Hybrid Ensemble Pipeline READY
📄 Retrieved 5 documents, context length: 8235 chars

💬 Câu hỏi: Hành vi điều khiển xe máy chở hai người có được phép theo quy định của pháp luật giao thông đường bộ Việt Nam không?
✍️  (Rewrite: "Hành vi điều khiển xe mô tô hai bánh chở theo hai người có được phép theo quy định của Luật Giao thông đường bộ Việt Nam không?")

📖 Câu trả lời:
Hành vi điều khiển xe máy chở hai người **chỉ được phép trong một số trường hợp ngoại lệ** theo quy định tại **Điều 33, khoản 1, Luật Trật tự, an toàn giao thông đường bộ số 36/2024/QH15**. Cụ thể, người lái xe mô tô hai bánh, xe gắn máy chỉ được chở tối đa hai người trong các trường hợp sau:

a) Chở người bệnh đi cấp cứu;  
b) Áp giải người có hành vi vi phạm pháp luật;  


In [3]:
# ==========================================
# CHECK QDRANT STATUS (DIAGNOSTIC)
# ==========================================
from pathlib import Path
import json
from qdrant_client import QdrantClient

qdrant_path = project_root / "data-ingestion" / "qdrant_db"
print(f"📁 Qdrant path: {qdrant_path}")
print(f"✅ Qdrant folder exists: {qdrant_path.exists()}\n")

# Check meta.json
meta_file = qdrant_path / "meta.json"
if meta_file.exists():
    with open(meta_file) as f:
        meta = json.load(f)
    if "traffic_law_add_context" in meta.get("collections", {}):
        col_info = meta["collections"]["traffic_law_add_context"]
        print(f"✅ Collection found in meta.json:")
        print(f"   - Vector size: {col_info['vectors']['size']}")
        print(f"   - Distance: {col_info['vectors']['distance']}")
    else:
        print(f"❌ Collection NOT in meta.json. Available: {list(meta.get('collections', {}).keys())}")
else:
    print(f"❌ meta.json not found at {meta_file}")

# Try direct Qdrant client
print("\n" + "="*70)
print("Testing QdrantClient...")
try:
    client = QdrantClient(path=str(qdrant_path), read_only=True)
    collection = client.get_collection("traffic_law_add_context")
    print(f"✅ QdrantClient CAN access collection:")
    print(f"   - Points: {collection.points_count}")
    print(f"   - Vector size: {collection.config.params.vectors.size}")
except Exception as e:
    print(f"❌ QdrantClient ERROR: {e}")

# Check retriever's Qdrant client
print("\n" + "="*70)
print("Testing retriever.qdrant_client...")
try:
    col_info = retriever.qdrant_client.get_collection("traffic_law_add_context")
    print(f"✅ retriever.qdrant_client CAN access collection")
    print(f"   - Points: {col_info.points_count}")
except Exception as e:
    print(f"❌ retriever.qdrant_client ERROR: {e}")

📁 Qdrant path: d:\GenAI\AIAgent\Agentic-RAG-Traffic-Law\data-ingestion\qdrant_db
✅ Qdrant folder exists: True

✅ Collection found in meta.json:
   - Vector size: 768
   - Distance: Cosine

Testing QdrantClient...
✅ QdrantClient CAN access collection:
   - Points: 3348
   - Vector size: 768

Testing retriever.qdrant_client...
❌ retriever.qdrant_client ERROR: Collection traffic_law_add_context not found
